# 17.1 持续预训练本节涵盖：领域适应预训练, 灾难性遗忘缓解

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ffrom copy import deepcopytorch.manual_seed(42)class SimpleModel(nn.Module):    def __init__(self):        super().__init__()        self.net = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))    def forward(self, x):        return self.net(x)model = SimpleModel()ref_model = deepcopy(model)# Train on general dataX_general = torch.randn(100, 10)y_general = torch.randint(0, 3, (100,))optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)for epoch in range(20):    loss = F.cross_entropy(model(X_general), y_general)    optimizer.zero_grad(); loss.backward(); optimizer.step()general_acc = (model(X_general).argmax(-1) == y_general).float().mean()print(f'General task accuracy before domain adaptation: {general_acc:.3f}')# Domain adaptation with EWC-like regularizationX_domain = torch.randn(100, 10) + 2.0y_domain = torch.randint(0, 3, (100,))# Compute Fisher information (importance of each parameter)fisher = {}for name, param in model.named_parameters():    fisher[name] = torch.zeros_like(param)model.train()for _ in range(10):    loss = F.cross_entropy(model(X_general), y_general)    optimizer.zero_grad(); loss.backward()    for name, param in model.named_parameters():        if param.grad is not None:            fisher[name] += param.grad.pow(2) / 10# Domain adaptation with EWC penaltyewc_lambda = 100optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)for epoch in range(20):    domain_loss = F.cross_entropy(model(X_domain), y_domain)    ewc_penalty = 0    for name, param in model.named_parameters():        ewc_penalty += (fisher[name] * (param - ref_model.state_dict()[name]).pow(2)).sum()    total_loss = domain_loss + ewc_lambda * ewc_penalty        optimizer.zero_grad(); total_loss.backward(); optimizer.step()domain_acc = (model(X_domain).argmax(-1) == y_domain).float().mean()general_acc_after = (model(X_general).argmax(-1) == y_general).float().mean()print(f'\nDomain task accuracy after adaptation: {domain_acc:.3f}')print(f'General task accuracy after adaptation: {general_acc_after:.3f}')print(f'\nEWC penalty preserves general knowledge while adapting to new domain.')